In [112]:
import sys
import os

# ensure we're always running from project root
os.chdir(os.path.dirname(os.getcwd()))
sys.path.insert(0, '.')

In [113]:
from src.load import load_data
from src.clean import clean_data
import pandas as pd

df_raw = load_data()
df = clean_data(df_raw)
max_date = df['InvoiceDate'].max()


no_guest_sales = df[df['Customer ID'].notna()][~df['Invoice'].str.startswith('C')]

── Data loaded ──────────────────────────
Rows:              1,033,019
Date range:        2009-12-01 → 2011-12-09
Unique customers:  5,940
Cancellations:     19,100
Null Customer IDs: 235,150
─────────────────────────────────────────
── Cleaning summary ─────────────────────
Rows before: 1,033,019
Rows after:  1,021,374
Removed:     11,645
Outlier rows flagged: 7
─────────────────────────────────────────


C:\Users\TomTurner\AppData\Local\Temp\ipykernel_23188\2603738903.py:10: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  no_guest_sales = df[df['Customer ID'].notna()][~df['Invoice'].str.startswith('C')]


In [114]:
no_guest_sales

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,TotalPrice,IsOutlier
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,83.40,False
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.00,False
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.00,False
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,100.80,False
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,30.00,False
...,...,...,...,...,...,...,...,...,...,...
1021369,581587,22613,PACK OF 20 SPACEBOY NAPKINS,12,2011-12-09 12:50:00,0.85,12680,France,10.20,False
1021370,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680,France,12.60,False
1021371,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680,France,16.60,False
1021372,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680,France,16.60,False


**RFM scoring**
- Recency — how recently did they last buy? Recent buyers are more valuable.
- Frequency — how many times have they bought? Loyal customers matter.
- Monetary — how much have they spent in total? Revenue concentration lives here.

In [115]:
customers = no_guest_sales['Customer ID'].unique()

**Recency** - days since most recent sale

In [116]:
# most recent purchase date
no_guest_sales[no_guest_sales['Customer ID'] == customers[0]]['InvoiceDate'].max()

Timestamp('2011-07-05 12:11:00')

In [117]:
# number of days since most recent purchase
(max_date - no_guest_sales[no_guest_sales['Customer ID'] == customers[2]]['InvoiceDate'].max()).days

448

In [118]:
df_recency = no_guest_sales.groupby(by='Customer ID', as_index=False)['InvoiceDate'].max()
df_recency.columns = ['CustomerID', 'LastPurchaseDate']
recent_date = df_recency['LastPurchaseDate'].max()
df_recency['Recency'] = df_recency['LastPurchaseDate'].apply(lambda x: (recent_date - x).days)
df_recency.head()

,CustomerID,LastPurchaseDate,Recency
0,12346,2011-01-18 10:01:00,325
1,12347,2011-12-07 15:52:00,1
2,12348,2011-09-25 13:13:00,74
3,12349,2011-11-21 09:51:00,18
4,12350,2011-02-02 16:01:00,309


**Frequency** - total number of orders

remove duplicate invoice numbers, counting number of invoices for unique customer

In [119]:
invoice_unique = no_guest_sales.drop_duplicates(subset=['Invoice'])
len(invoice_unique[invoice_unique['Customer ID'] == customers[0]])

8

In [ ]:
frequency_df = no_guest_sales.groupby(by=['Customer ID'], as_index=False)['Invoice'].count()
frequency_df.columns = ['CustomerID', 'Frequency']
frequency_df.head()

,CustomerID,Frequency
0,12346,25
1,12347,222
2,12348,46
3,12349,172
4,12350,16


**Monetary** - total revenue generated by customer

In [121]:
# keeping cancelations
no_guest_df = df[df['Customer ID'].notna()]

In [122]:
no_guest_df[no_guest_df['Customer ID'] == customers[0]]['TotalPrice'].sum()

np.float64(2289.58)

In [123]:
monetary_df = no_guest_df.groupby(by='Customer ID', as_index=False)['TotalPrice'].sum()
monetary_df.columns = ['CustomerID', 'Monetary']
monetary_df.head()

,CustomerID,Monetary
0,12346,169.36
1,12347,4921.53
2,12348,1658.40
3,12349,3654.54
4,12350,294.40


In [124]:
rf_df = df_recency.merge(frequency_df, on='CustomerID')
rfm_df = rf_df.merge(monetary_df, on='CustomerID').drop(columns='LastPurchaseDate')
rfm_df.head()

,CustomerID,Recency,Frequency,Monetary
0,12346,325,25,169.36
1,12347,1,222,4921.53
2,12348,74,46,1658.40
3,12349,18,172,3654.54
4,12350,309,16,294.40


In [ ]:
rfm_df['R_rank'] = rfm_df['Recency'].rank(ascending=False)
rfm_df['F_rank'] = rfm_df['Frequency'].rank(ascending=True)
rfm_df['M_rank'] = rfm_df['Monetary'].rank(ascending=True)

rfm_df['R_rank_norm'] = (rfm_df['R_rank'] / rfm_df['R_rank'].max())
rfm_df['F_rank_norm'] = (rfm_df['F_rank'] / rfm_df['F_rank'].max())
rfm_df['M_rank_norm'] = (rfm_df['M_rank'] / rfm_df['M_rank'].max())

rfm_df.drop(columns=['R_rank', 'F_rank', 'M_rank'], inplace=True)

rfm_df_quart = rfm_df.copy()

rfm_df_quart.head()

,CustomerID,Recency,Frequency,Monetary,R_rank_norm,F_rank_norm,M_rank_norm
0,12346,325,25,169.36,0.289356,0.296309,0.113807
1,12347,1,222,4921.53,0.984585,0.846036,0.893882
2,12348,74,46,1658.40,0.545815,0.464200,0.684894
3,12349,18,172,3654.54,0.808991,0.798787,0.849282
4,12350,309,16,294.40,0.301498,0.194976,0.215140


Creating quartiles

In [ ]:
# 1=low, 4=high
rfm_df_quart['R_rank_quart'] = pd.qcut(rfm_df_quart['R_rank_norm'], 4, labels=False) + 1
rfm_df_quart['F_rank_quart'] = pd.qcut(rfm_df_quart['F_rank_norm'], 4, labels=False) + 1
rfm_df_quart['M_rank_quart'] = pd.qcut(rfm_df_quart['M_rank_norm'], 4, labels=False) + 1

rfm_df_quart['RFM_Score'] = rfm_df_quart['R_rank_quart'] + rfm_df_quart['F_rank_quart'] + rfm_df_quart['M_rank_quart']

rfm_df_quart.head()

,CustomerID,Recency,Frequency,Monetary,R_rank_norm,F_rank_norm,M_rank_norm,R_rank_quart,F_rank_quart,M_rank_quart,RFM_Score
0,12346,325,25,169.36,0.289356,0.296309,0.113807,2,2,1,5
1,12347,1,222,4921.53,0.984585,0.846036,0.893882,4,4,4,12
2,12348,74,46,1658.40,0.545815,0.464200,0.684894,3,2,3,8
3,12349,18,172,3654.54,0.808991,0.798787,0.849282,4,4,4,12
4,12350,309,16,294.40,0.301498,0.194976,0.215140,2,1,1,4


In [129]:
rfm_df_quart.sort_values('RFM_Score', ascending=False).head(10)

,CustomerID,Recency,Frequency,Monetary,R_rank_norm,F_rank_norm,M_rank_norm,R_rank_quart,F_rank_quart,M_rank_quart,RFM_Score
34,12380,21,267,7171.00,0.788495,0.879101,0.932843,4,4,4,12
3,12349,18,172,3654.54,0.808991,0.798787,0.849282,4,4,4,12
1140,13510,0,289,4488.39,1.000000,0.890294,0.882262,4,4,4,12
3960,16369,17,167,2654.48,0.819497,0.794087,0.789474,4,4,4,12
1165,13536,0,178,3444.39,1.000000,0.806391,0.839713,4,4,4,12
1163,13534,1,392,7324.50,0.984585,0.931904,0.934211,4,4,4,12
3951,16360,4,636,3349.57,0.937478,0.968558,0.834757,4,4,4,12
3949,16358,0,154,2917.10,1.000000,0.776230,0.806220,4,4,4,12
3944,16353,3,173,9781.93,0.952463,0.800325,0.956767,4,4,4,12
1137,13507,2,162,2479.16,0.969600,0.787679,0.776316,4,4,4,12


In [159]:
rfm_df.columns

Index(['CustomerID', 'Recency', 'Frequency', 'Monetary', 'R_rank_norm',
       'F_rank_norm', 'M_rank_norm', 'RFM_Score'],
      dtype='str')

In [155]:
top_customers = list(rfm_df_quart[rfm_df_quart['RFM_Score'] >= rfm_df_quart['RFM_Score'].quantile(.8)]['CustomerID'])
top_customers_df = df[df["Customer ID"].isin(top_customers)]
print(f"Total revenue by top customers: {top_customers_df['TotalPrice'].sum()}")

Total revenue by top customers: 12996351.108


In [157]:
(top_customers_df['TotalPrice'].sum() / df['TotalPrice'].sum()) * 100

np.float64(68.6841820501616)

12.9 million (68.7% of total revenue) from top 20% customers

## Using weights

In [149]:
rfm_weights = {
    'r': 0.15,
    'f': 0.28,
    'm': 0.57,
    'norm': 0.05
}

rfm_df['RFM_Score'] = rfm_weights['r'] * rfm_df['R_rank_norm'] + rfm_weights['f'] * rfm_df['F_rank_norm'] + rfm_weights['m'] * rfm_df['M_rank_norm']

In [150]:
rfm_df.sort_values('RFM_Score', ascending=False).head(10)

,CustomerID,Recency,Frequency,Monetary,R_rank_norm,F_rank_norm,M_rank_norm,RFM_Score
2520,14911,0,10909,258522.43,1.00,1.00,1.00,1.0000
2916,15311,0,4303,111444.36,1.00,1.00,1.00,1.0000
5666,18102,0,1040,578408.64,1.00,0.99,1.00,0.9972
2262,14646,1,3819,523202.74,0.98,1.00,1.00,0.9970
5408,17841,1,12425,67287.06,0.98,1.00,1.00,0.9970
730,13089,2,3315,109893.24,0.97,1.00,1.00,0.9955
5085,17511,2,1868,168504.44,0.97,1.00,1.00,0.9955
1423,13798,0,721,73177.50,1.00,0.98,1.00,0.9944
393,12748,0,6557,46674.18,1.00,1.00,0.99,0.9943
2222,14606,0,6347,28913.22,1.00,1.00,0.99,0.9943


In [ ]:
top_customers_weight = list(rfm_df[rfm_df['RFM_Score'] >= rfm_df['RFM_Score'].quantile(.8)]['CustomerID'])
top_customers_weight_df = df[df["Customer ID"].isin(top_customers_weight)]
print(f"Total revenue by top customers: {top_customers_weight_df['TotalPrice'].sum()}")

Total revenue by top customers: 12160001.269000001
